# Pipeline completo: del CSV al riesgo por pedido y al recomendador

Este notebook corre el proyecto de punta a punta llamando al código del repositorio (`src/`). No reimplementa nada: cada sección usa la misma función que usan los scripts, la app y la API, así que sus cifras son las del README y el informe.

**Qué hace, en orden:** carga los datos, construye el target, arma las variables sin fuga, separa por tiempo, compara modelos, calibra y fija los niveles de riesgo, monitorea el nivel alto, predice como la API y evalúa el sistema de recomendación.

**Requisitos:** los 9 CSV de Kaggle en `data/raw/` y las dependencias de `requirements.txt`. Tarda unos minutos, sobre todo la comparación de modelos. **No escribe ningún archivo:** el modelo se reentrena en memoria y se compara con el guardado en `models/`, si existe.

In [1]:
import os
import sys
import warnings
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.width", 140)
print("Raíz del repositorio:", RAIZ.name)

Raíz del repositorio: proyecto-final-


## 1. Datos

Las 9 tablas públicas de Olist. `load_raw_tables` carga las que necesita el modelo y no carga las reseñas a propósito: son posteriores a la entrega y no pueden entrar al modelo (decisión 4).

In [2]:
from src.data_loading import load_raw_tables
from src.feature_engineering import load_geolocation

tablas = {**load_raw_tables(), "geolocation": load_geolocation()}
pd.DataFrame({"filas": {nombre: len(t) for nombre, t in tablas.items()}})

,filas
orders,99441
items,112650
products,32951
sellers,3095
customers,99441
payments,103886
geolocation,1000163


## 2. Target y período de estudio

Un pedido está retrasado si se entrega al menos un día calendario después de la fecha estimada (decisión 2). Se usan los pedidos entregados comprados entre enero de 2017 y agosto de 2018 (decisión 8): en 2016 hay meses casi vacíos, y septiembre y octubre de 2018 tienen 20 pedidos, casi todos cancelados.

In [3]:
from src.target import TARGET_COL, build_target

pedidos = build_target(tablas["orders"])
print(f"Pedidos en el dataset: {len(tablas['orders']):,}")
print(f"Pedidos con target (entregados, del período): {len(pedidos):,}")
print(f"Tasa de retraso: {pedidos[TARGET_COL].mean():.2%}")
anio = pd.to_datetime(pedidos["order_purchase_timestamp"]).dt.year
pedidos.groupby(anio)[TARGET_COL].agg(pedidos="size", tasa_retraso="mean")

Pedidos en el dataset: 99,441
Pedidos con target (entregados, del período): 96,203
Tasa de retraso: 6.79%


,pedidos,tasa_retraso
order_purchase_timestamp,,
2017,43426,0.0565
2018,52777,0.0773


## 3. Variables, sin fuga de datos

`build_model_dataset` arma una fila por pedido con lo que se conoce al aprobar el pago. El modelo principal usa las variables del baseline sin el mes de compra, más la distancia cliente-vendedor y los feriados de los 7 días siguientes. Las fechas de despacho y de entrega se usan solo para el target.

In [4]:
from src.feature_engineering import build_model_dataset
from src.training import selected_feature_lists

dataset = build_model_dataset(tablas)
numericas, categoricas = selected_feature_lists()
print(f"Pedidos para modelar: {len(dataset):,} (sin los entregados que no tienen fecha de aprobación)")
print("Variables numéricas:", numericas)
print("Variables categóricas:", categoricas)
prohibidas = {"order_delivered_customer_date", "order_delivered_carrier_date", "review_score", "order_status"}
assert not prohibidas & set(numericas + categoricas), "Hay una variable con fuga"
print("Control de fuga: ninguna variable posterior a la aprobación del pago.")

Pedidos para modelar: 96,189 (sin los entregados que no tienen fecha de aprobación)
Variables numéricas: ['purchase_dayofweek', 'purchase_hour', 'promised_days', 'n_items', 'n_sellers', 'total_price', 'total_freight', 'freight_ratio', 'total_weight_g', 'total_volume_cm3', 'max_installments', 'approval_hours', 'max_distance_km', 'n_national_holidays_next_7d']
Variables categóricas: ['customer_state', 'main_payment_type', 'seller_other_state']
Control de fuga: ninguna variable posterior a la aprobación del pago.


## 4. Validación temporal

Se entrena con el pasado y se evalúa con meses que el modelo nunca vio, como en la operación real. La tasa de retraso cambia mucho entre períodos, por eso un split al azar daría métricas optimistas.

In [5]:
from src.features import PREDICTION_TIME_COL
from src.validation import temporal_three_way_split

train, validacion, test = temporal_three_way_split(dataset)
pd.DataFrame(
    {
        nombre: {
            "desde": parte[PREDICTION_TIME_COL].min().date(),
            "hasta": parte[PREDICTION_TIME_COL].max().date(),
            "pedidos": len(parte),
            "tasa_retraso": parte[TARGET_COL].mean(),
        }
        for nombre, parte in [("train", train), ("validación", validacion), ("test", test)]
    }
).T

,desde,hasta,pedidos,tasa_retraso
train,2017-01-05,2018-03-19,61276,0.0783
validación,2018-03-20,2018-05-26,15620,0.0681
test,2018-05-27,2018-08-29,19293,0.0349


## 5. Comparación de modelos

Validación cruzada temporal de 4 tramos sobre train + validación. Se elige por PR-AUC, porque con pocos retrasos la ROC-AUC se ve optimista; si dos modelos quedan a menos de 0,01, gana el más simple. El test se mira solo al final, como diagnóstico.

In [6]:
from src.training import compare_models

reporte_modelos, _ = compare_models(dataset)
pd.DataFrame(
    {
        nombre: {
            "PR-AUC (CV)": reporte_modelos["cross_validation"][nombre]["pr_auc"],
            "ROC-AUC (CV)": reporte_modelos["cross_validation"][nombre]["roc_auc"],
            "PR-AUC (test)": reporte_modelos["test"][nombre]["pr_auc"],
            "ROC-AUC (test)": reporte_modelos["test"][nombre]["roc_auc"],
        }
        for nombre in reporte_modelos["cross_validation"]
    }
).T

,PR-AUC (CV),ROC-AUC (CV),PR-AUC (test),ROC-AUC (test)
logistica_baseline,0.2064,0.7013,0.0719,0.7003
logistica_seleccion,0.2114,0.7106,0.0734,0.7080
arboles_seleccion,0.2156,0.7101,0.0518,0.5788
arboles_completa,0.2076,0.7063,0.0462,0.5570


In [7]:
print("Modelo elegido:", reporte_modelos["selected_model"])

Modelo elegido: logistica_seleccion


## 6. Calibración y niveles de riesgo

El modelo se entrena con pesos de clase balanceados: ordena bien, pero infla las probabilidades. La calibración de Platt, ajustada en validación, las lleva a la escala real sin cambiar el orden. Los niveles se fijan por volumen en validación: alto es el 5 % más riesgoso y medio el 15 % siguiente. Después el modelo se reentrena con train + validación (`run` de `src/umbral.py`, lo mismo que `python -m src.umbral`, pero sin guardar).

In [8]:
from src.umbral import CALIBRATED_MODEL_PATH, run

reporte, artefacto = run(tablas)
for cohorte in ("validacion", "test", "test_reentrenado"):
    r = reporte["cohortes"][cohorte]
    print(f"{cohorte:17s} ROC-AUC {r['roc_auc']:.3f} · PR-AUC {r['pr_auc']:.3f} · prevalencia {r['prevalencia']:.1%}"
          f" · riesgo medio sin calibrar {r['riesgo_medio_sin_calibrar']:.1%} · calibrado {r['riesgo_medio_calibrado']:.1%}")
print("\nCortes de riesgo (fijados en validación):", {k: round(v, 4) for k, v in artefacto["cuts"].items()})
pd.DataFrame(reporte["cohortes"]["test_reentrenado"]["niveles"]).set_index("nivel")

validacion        ROC-AUC 0.735 · PR-AUC 0.175 · prevalencia 6.8% · riesgo medio sin calibrar 44.7% · calibrado 6.8%
test              ROC-AUC 0.708 · PR-AUC 0.073 · prevalencia 3.5% · riesgo medio sin calibrar 49.5% · calibrado 9.1%
test_reentrenado  ROC-AUC 0.707 · PR-AUC 0.073 · prevalencia 3.5% · riesgo medio sin calibrar 49.2% · calibrado 9.3%

Cortes de riesgo (fijados en validación): {'alto': 0.1966, 'medio': 0.0946}


,pct_pedidos,tasa_retraso,pct_retrasos,lift
nivel,,,,
alto,0.1127,0.0741,0.2392,2.1230
medio,0.2197,0.0561,0.3536,1.6095
bajo,0.6676,0.0213,0.4071,0.6098


Si existe el modelo guardado en `models/`, se comprueba que el que se acaba de entrenar sea el mismo: mismos cortes y mismas probabilidades.

In [9]:
if CALIBRATED_MODEL_PATH.exists():
    guardado = joblib.load(CALIBRATED_MODEL_PATH)
    from src.umbral import predict_risk
    dif = np.abs(predict_risk(guardado, test)["prob_retraso"] - predict_risk(artefacto, test)["prob_retraso"]).max()
    print("Cortes del modelo guardado:", {k: round(v, 4) for k, v in guardado["cuts"].items()})
    print(f"Máxima diferencia de probabilidad en test: {dif:.2e}")
else:
    print("No hay modelo guardado: se usa el que se acaba de entrenar.")

Cortes del modelo guardado: {'alto': 0.1966, 'medio': 0.0946}
Máxima diferencia de probabilidad en test: 0.00e+00


## 7. Monitoreo mensual

Con los cortes fijos, se mira cada mes qué parte de los pedidos cae en el nivel alto y si ese nivel sigue concentrando retrasos (lift). La razón entre retrasos observados y esperados (O/E) mide si la probabilidad sigue calibrada.

In [10]:
from src.monitoreo import monthly_high_risk
from src.umbral import predict_risk

puntuados = test[[PREDICTION_TIME_COL, TARGET_COL]].join(predict_risk(artefacto, test))
monitoreo = pd.DataFrame(monthly_high_risk(puntuados)).set_index("mes")
mes = puntuados[PREDICTION_TIME_COL].dt.to_period("M").astype(str)
o_e = puntuados.groupby(mes).apply(lambda g: g[TARGET_COL].sum() / g["prob_retraso"].sum())
monitoreo.join(o_e.rename("O/E"))[["pedidos", "pct_alto", "tasa_retraso_alto", "tasa_retraso_mes", "lift_alto", "O/E"]]

,pedidos,pct_alto,tasa_retraso_alto,tasa_retraso_mes,lift_alto,O/E
mes,,,,,,
2018-06,6094,0.0203,0.1048,0.0117,8.9984,0.2395
2018-07,6047,0.1399,0.0686,0.0319,2.1480,0.3018
2018-08,6504,0.1851,0.0748,0.0627,1.1916,0.4806


El nivel alto pasa del 2 % de los pedidos en junio al 18,5 % en agosto, contra el 5 % de diseño, y el O/E queda muy por debajo de 1: el modelo predice más retrasos de los que hay porque la prevalencia bajó y Olist acortó los plazos. El lift se mantiene por encima de 1: sigue ordenando. Es la señal para recalibrar con pedidos recientes.

## 8. La misma predicción que la API

La app y la API llaman a `predict_risk` con el modelo guardado. Estos son los 3 pedidos reales del período de test que se usan en la demo (`scripts/demo_pedidos.py`).

In [11]:
demo = ["ae213a9f84c777fb7a31e8c0f09fd30c", "2b5d5c1847d069c5efeb82f4056c15d8", "3891073d9934c3bc761f151f90f1be17"]
por_pedido = dataset.set_index("order_id").loc[demo]
predict_risk(artefacto, por_pedido).join(por_pedido[["customer_state", "promised_days", TARGET_COL]])

,prob_retraso,nivel_riesgo,customer_state,promised_days,pedido_retrasado
order_id,,,,,
ae213a9f84c777fb7a31e8c0f09fd30c,0.5465,alto,AL,28,1
2b5d5c1847d069c5efeb82f4056c15d8,0.0205,bajo,SP,13,0
3891073d9934c3bc761f151f90f1be17,0.3053,alto,RJ,21,0


## 9. Sistema de recomendación

En Olist el 97 % de los clientes compra una sola vez, así que se recomienda por el contexto de la compra: en cada categoría y estado del cliente, los 3 vendedores con menor riesgo medio según el clasificador. Se mide en el período de test contra tres referencias: historial del vendedor, popularidad y azar (decisión 12, `docs/recomendador.md`).

In [12]:
from src.recomendador import (
    evaluar_recomendacion,
    ic_diferencia,
    preparar_clasificador,
    puntaje_vendedores,
    recomendar,
    resumen_evaluacion,
)

pedidos_vend, inicio_test = preparar_clasificador(tablas, artefacto)
vendedores = puntaje_vendedores(pedidos_vend, inicio_test)
evaluacion = evaluar_recomendacion(pedidos_vend, vendedores, inicio_test, k=3)
print(f"{len(vendedores)} vendedores con historial · {len(evaluacion)} combinaciones de categoría y estado medidas")
resumen_evaluacion(evaluacion)

987 vendedores con historial · 211 combinaciones de categoría y estado medidas


,precision,recall,retraso
clasificador,0.7125,0.2828,0.0257
historial,0.6983,0.2839,0.0279
popularidad,0.6935,0.2774,0.0275
azar,0.6612,0.2696,0.0314


In [13]:
for a, b in [("clasificador", "azar"), ("clasificador", "historial")]:
    d, lo, hi = ic_diferencia(evaluacion, a, b)
    print(f"Retraso de los recomendados, {a} − {b}: {d * 100:+.2f} pp (IC 95 % {lo * 100:+.2f} a {hi * 100:+.2f})")

Retraso de los recomendados, clasificador − azar: -0.58 pp (IC 95 % -0.85 a -0.31)
Retraso de los recomendados, clasificador − historial: -0.23 pp (IC 95 % -0.59 a +0.12)


Un ejemplo de lo que vería el cliente: cama, mesa y baño para Río de Janeiro.

In [14]:
recomendar(pedidos_vend, vendedores, inicio_test, "cama_mesa_banho", "RJ", k=3)

,ventas_aqui,precio_mediano,riesgo_predicho,pedidos_hist
seller_id,,,,
6fd52c528dcb38be2eea044946b811f8,11,105.6000,0.0280,58
dd2bdf855a9172734fbc3744021ae9b9,14,69.9000,0.0377,103
d2374cbcbb3ca4ab1086534108cc3ab7,48,27.4000,0.0395,406


## Resultado

- **Clasificador:** ROC-AUC 0,707 y PR-AUC 0,073 en test, el doble del piso de 0,035. El nivel alto reúne el 11,2 % de los pedidos y el 24,1 % de los retrasos (lift 2,15).
- **Monitoreo:** el nivel alto se desvía mes a mes y el O/E queda bajo 1; el orden se mantiene. Se recalibra todos los meses.
- **Recomendador:** los vendedores recomendados se retrasan 2,6 % contra 3,1 % al azar; no se distingue del historial. Es un efecto modesto: 7 de cada 10 retrasos ocurren en el transporte.

Los límites y las decisiones están en el README, en `docs/decisiones.md` y en el informe final.